# Lang — free language services on Google Colab

This notebook runs free, open-source services for the Lang app on a Colab GPU:

| Service | What it does in the app | Engine |
|---|---|---|
| **LLM** | word definitions + IPA, sentence translation, example sentences, grading translations | [Ollama](https://ollama.com) with an open model (default `qwen2.5:7b`) |
| **Speech recognition** | speaking minigames | [faster-whisper](https://github.com/SYSTRAN/faster-whisper) |
| **Text to speech** | listening minigames | Microsoft neural voices via [edge-tts](https://github.com/rany2/edge-tts) |

Everything is exposed behind **one public URL** (a free Cloudflare quick tunnel) with OpenAI-compatible endpoints.

**How to use**
1. `Runtime → Change runtime type → T4 GPU` (a GPU makes the LLM ~20× faster).
2. `Runtime → Run all`.
3. Copy the URL printed at the end into **Lang → Configuration → Services → Colab URL**, and choose *Google Colab* for the services you want.
4. Keep this tab open while you use the app. The URL changes every time the notebook restarts.

No tunnel allowed on your network? Use the **offline batches** section at the bottom instead: download a job file from the app, run it here, upload the results back.

In [ ]:
# Settings
MODEL = "qwen2.5:7b"          # any Ollama model: qwen2.5:7b, gemma2:9b, llama3.1:8b, mistral-nemo...
NUM_CTX = 8192                # context window used for batches
WHISPER_MODEL = "small"       # tiny, base, small, medium, large-v3 (bigger = more accurate, slower)
ENABLE_SPEECH_RECOGNITION = True
ENABLE_TEXT_TO_SPEECH = True
PORT = 8000

In [ ]:
# Install Ollama, cloudflared and the Python packages (about 2 minutes)
!sudo apt-get update -y > /dev/null 2>&1
!sudo apt-get install -y zstd lshw > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
!pip install -q fastapi uvicorn httpx python-multipart edge-tts faster-whisper
print("Installed.")

In [ ]:
# Start Ollama and download the model
import os, subprocess, time, requests

env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_KEEP_ALIVE="-1", OLLAMA_ORIGINS="*")
subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        break
    except Exception:
        time.sleep(1)
print(f"Downloading {MODEL} (first time only, a few minutes)...")
!ollama pull {MODEL} > /dev/null 2>&1
print("Ollama is ready with", MODEL)

In [ ]:
%%writefile /content/lang_server.py
# OpenAI-compatible server used by the Lang app.
import asyncio, os, random, tempfile
import httpx
from fastapi import FastAPI, File, Form, HTTPException, Request, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response

OLLAMA = "http://127.0.0.1:11434"
MODEL = os.environ.get("LANG_MODEL", "qwen2.5:7b")
NUM_CTX = int(os.environ.get("LANG_NUM_CTX", "8192"))
WHISPER_MODEL = os.environ.get("LANG_WHISPER", "small")

app = FastAPI(title="Lang Colab services")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])


@app.get("/")
@app.get("/v1/health")
async def health():
    return {"ok": True, "model": MODEL}


@app.get("/v1/models")
async def models():
    return {"object": "list", "data": [{"id": MODEL, "object": "model"}]}


@app.post("/v1/chat/completions")
async def chat(request: Request):
    body = await request.json()
    payload = {
        "model": body.get("model") or MODEL,
        "messages": body["messages"],
        "stream": False,
        "options": {"temperature": body.get("temperature", 0.2), "num_ctx": NUM_CTX},
    }
    async with httpx.AsyncClient(timeout=900) as client:
        reply = await client.post(f"{OLLAMA}/api/chat", json=payload)
    if reply.status_code != 200:
        raise HTTPException(reply.status_code, reply.text)
    content = reply.json()["message"]["content"]
    return {
        "id": "chatcmpl-lang",
        "object": "chat.completion",
        "model": payload["model"],
        "choices": [{"index": 0, "message": {"role": "assistant", "content": content}, "finish_reason": "stop"}],
    }


_whisper = None


def whisper():
    global _whisper
    if _whisper is None:
        from faster_whisper import WhisperModel
        try:
            _whisper = WhisperModel(WHISPER_MODEL, device="cuda", compute_type="float16")
        except Exception:
            _whisper = WhisperModel(WHISPER_MODEL, device="cpu", compute_type="int8")
    return _whisper


@app.post("/v1/audio/transcriptions")
async def transcriptions(
    file: UploadFile = File(...),
    model: str = Form("whisper"),
    language: str = Form(None),
    response_format: str = Form("json"),
):
    data = await file.read()
    suffix = os.path.splitext(file.filename or "speech.webm")[1] or ".webm"
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as handle:
        handle.write(data)
        path = handle.name

    def run():
        segments, _ = whisper().transcribe(path, language=language or None, beam_size=5, vad_filter=True)
        return " ".join(segment.text.strip() for segment in segments).strip()

    try:
        text = await asyncio.to_thread(run)
    finally:
        os.unlink(path)
    return {"text": text}


_voices = None


async def voice_for(locale):
    import edge_tts
    global _voices
    if _voices is None:
        _voices = await edge_tts.list_voices()
    wanted = (locale or "en-US").lower().replace("_", "-")
    exact = [v["ShortName"] for v in _voices if v["Locale"].lower() == wanted]
    same_language = [v["ShortName"] for v in _voices if v["Locale"].lower().split("-")[0] == wanted.split("-")[0]]
    pool = exact or same_language
    if not pool:
        raise HTTPException(400, f"No neural voice for {locale}")
    # A little variety between voices (e.g. male / female) helps listening.
    return random.choice(pool[:4])


@app.post("/v1/audio/speech")
async def speech(request: Request):
    import edge_tts
    body = await request.json()
    text = (body.get("input") or "").strip()
    if not text:
        raise HTTPException(400, "Empty text")
    voice = body.get("voice") or ""
    if "-" not in voice:  # OpenAI style names ("alloy") or empty: pick by language
        voice = await voice_for(body.get("language"))
    audio = b""
    async for chunk in edge_tts.Communicate(text, voice).stream():
        if chunk["type"] == "audio":
            audio += chunk["data"]
    return Response(content=audio, media_type="audio/mpeg")

In [ ]:
# Start the server and warm up the optional services
import subprocess, time, requests

server_env = dict(os.environ, LANG_MODEL=MODEL, LANG_NUM_CTX=str(NUM_CTX), LANG_WHISPER=WHISPER_MODEL)
subprocess.Popen(
    ["python", "-m", "uvicorn", "lang_server:app", "--host", "127.0.0.1", "--port", str(PORT)],
    cwd="/content", env=server_env, stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(60):
    try:
        requests.get(f"http://127.0.0.1:{PORT}/v1/health", timeout=2)
        break
    except Exception:
        time.sleep(1)

reply = requests.post(f"http://127.0.0.1:{PORT}/v1/chat/completions",
                      json={"messages": [{"role": "user", "content": "Reply with: ready"}]}, timeout=600)
print("LLM:", reply.json()["choices"][0]["message"]["content"].strip()[:60])
if ENABLE_TEXT_TO_SPEECH:
    audio = requests.post(f"http://127.0.0.1:{PORT}/v1/audio/speech", json={"input": "hola", "language": "es-ES"}, timeout=120)
    print("Text to speech:", "ok" if audio.ok else audio.text[:200])
if ENABLE_SPEECH_RECOGNITION:
    import lang_server  # downloads the Whisper model once
    lang_server.whisper()
    print("Speech recognition: ok")

In [ ]:
# Open the public tunnel and print the URL for the app
import re, subprocess, time

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
url = None
start = time.time()
while time.time() - start < 90 and not url:
    line = tunnel.stdout.readline()
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
if url:
    print("=" * 70)
    print("Paste this URL into Lang → Configuration → Services → Colab URL:")
    print()
    print("   ", url)
    print()
    print("Model:", MODEL)
    print("=" * 70)
else:
    print("The tunnel did not start. Use the offline batches below instead.")

## Offline batches (no tunnel needed)

1. In the app: **Configuration → Services → Google Colab → Offline batches** → download a *definitions* or *sentence translation* job file.
2. Run the cell below and upload the job file(s). Results are processed in batches and downloaded automatically.
3. In the app, click **Upload results**. They are saved to the database.

In [ ]:
import json, re, requests
from google.colab import files

BATCH = {"definitions": 40, "translations": 20}


def ask(prompt):
    reply = requests.post("http://127.0.0.1:11434/api/chat", json={
        "model": MODEL, "stream": False, "format": "json",
        "messages": [{"role": "user", "content": prompt}],
        "options": {"temperature": 0.1, "num_ctx": NUM_CTX},
    }, timeout=900)
    text = reply.json()["message"]["content"]
    data = json.loads(text[text.find("{"):text.rfind("}") + 1])
    return data.get("items", [])


def definitions_prompt(language, words):
    return (f"For each {language} word below (taken from a frequency list of real texts, so it may be an inflected form), give:\n"
            '- "english": 1-4 short English translations, most common meaning first. Translate the form itself.\n'
            '- "pos": part of speech\n- "ipa": pronunciation in IPA, without slashes\n'
            'Return only JSON: {"items":[{"word":"...","english":["..."],"pos":"...","ipa":"..."}]} with one item per word, in the same order.\n'
            f"Words: {json.dumps(words, ensure_ascii=False)}")


def translations_prompt(language, sentences):
    items = [{"id": i + 1, "text": s} for i, s in enumerate(sentences)]
    return (f"Translate each {language} sentence into natural, faithful English. Do not add explanations.\n"
            'Return only JSON: {"items":[{"id":1,"english":"..."}]} with one item per sentence.\n'
            + json.dumps(items, ensure_ascii=False))


for name, data in files.upload().items():
    job = json.loads(data)
    kind, language, items = job["type"], job["language"], job["items"]
    results = []
    size = BATCH[kind]
    for start in range(0, len(items), size):
        batch = items[start:start + size]
        print(f"{name}: {start}/{len(items)}")
        try:
            if kind == "definitions":
                answers = ask(definitions_prompt(language, [item["text"] for item in batch]))
                by_word = {str(a.get("word", "")).lower(): a for a in answers}
                for index, item in enumerate(batch):
                    answer = by_word.get(item["text"].lower()) or (answers[index] if len(answers) == len(batch) else None)
                    if answer:
                        results.append({"id": item["id"], "english": answer.get("english", []),
                                        "ipa": answer.get("ipa"), "pos": answer.get("pos")})
            else:
                answers = ask(translations_prompt(language, [item["text"] for item in batch]))
                for answer in answers:
                    index = int(answer.get("id", 0)) - 1
                    if 0 <= index < len(batch) and answer.get("english"):
                        results.append({"id": batch[index]["id"], "english": answer["english"]})
        except Exception as error:
            print("  batch failed:", error)
    out = name.replace("-job", "-results")
    with open(out, "w") as handle:
        json.dump({"format": "lang-colab-results", "type": kind, "model": MODEL, "items": results}, handle, ensure_ascii=False)
    print(f"{len(results)} results → {out}")
    files.download(out)